# 7. 트리 모델, 앙상블과 차원 축소

- 목표: 결정트리, 랜덤포레스트, 부스팅, PCA, LDA를 묶어 모델 확장 감각을 기릅니다.
- 흐름: 자전거 수요 예측 베이스라인으로 회귀 프로젝트 흐름을 확인합니다.


# 결정 트리와 앙상블 기법

**결정 트리와 앙상블 기법**은 머신러닝에서 가장 널리 쓰이는 알고리즘 중 하나입니다.  
- **결정 트리(Decision Tree)**: 구조가 직관적이고 시각화가 쉬워 해석 가능성이 높음. 그러나 단독으로는 과적합 위험이 크고 성능이 제한적일 수 있음.  
- **앙상블 기법(Ensemble Methods)**: 여러 개의 모델을 결합하여 단일 모델보다 더 좋은 성능을 발휘. 특히 배깅(Bagging)과 부스팅(Boosting)이 대표적.  

👉 이 토픽에서는 **결정 트리 기본기**부터 **랜덤 포레스트, 부스팅(XGBoost 포함)** 까지 학습하여, 복잡한 데이터셋에서도 효과적인 모델을 구축할 수 있도록 합니다.  

학습 목표

- 결정 트리의 기본 개념과 작동 방식을 이해하고 설명할 수 있다.  
- 앙상블 기법의 원리를 이해하고 실제 데이터에 적용할 수 있다.  
- 결정 트리 및 앙상블 모델을 학습·평가하고, 성능을 비교할 수 있다. 

## 1. 들어가기

머신러닝 모델 중에는 복잡하지만 해석하기 어려운 모델(예: 신경망)과  
단순하지만 이해하기 쉬운 모델(예: 선형 회귀)이 있습니다.  

👉 **결정 트리(Decision Tree)** 는 그 중간에 위치합니다.  
- 데이터의 규칙을 "나무(tree)" 구조로 표현  
- 사람이 이해하기 쉽고, 시각화 가능  
- 그러나 깊게 만들면 과적합이 쉽게 발생  

  <img src="image/Decision Tree.webp" width="500">

이미지 출처 : https://www.displayr.com/what-is-a-decision-tree/

이번 장에서는 **결정 트리의 기본 개념**부터 실습까지 다룹니다.


## 2. 결정 트리 (Decision Tree)

### 2.1 기본 개념
- 데이터를 분할하여 예측하는 알고리즘  
- 트리 구조:  
  - **루트 노드(root)**: 시작점  
  - **분기 노드(split node)**: 질문을 던지는 부분 (예: 나이 > 30?)  
  - **리프 노드(leaf node)**: 최종 예측 결과 (예: 생존 / 사망)  

  <img src="image/tree_arch.webp">  
  
  이미지 출처 : https://medium.com/@ryassminh/math-for-ml-understand-regression-decision-trees-with-simple-examples-9474fceb6802


### 2.2 작동 방식
1. 데이터를 가장 잘 나누는 기준(feature, 조건)을 선택  
2. 해당 조건으로 분할  
3. 각 분할된 그룹에 대해 다시 같은 과정을 반복  
4. 더 이상 나눌 수 없을 때 예측 결과 결정  

👉 즉, "질문을 반복하면서 최종 답을 찾는 과정"


### 2.3 트리는 어떤 질문을 고를까?

트리는 여러 질문 후보를 비교하면서 데이터를 나눕니다.  
어려운 공식보다 다음 질문만 기억하면 됩니다.

> “이 질문으로 나눴을 때, 생존한 승객과 생존하지 못한 승객이 더 잘 구분되는가?”

예시: 타이타닉 생존 예측
- `성별 == 여성인가?`  
  - 여성 승객 쪽에 생존자가 더 많이 모이면 좋은 질문입니다.
- `Pclass <= 2인가?`  
  - 1·2등석 승객과 3등석 승객의 생존 패턴이 다르면 좋은 질문입니다.
- `나이 <= 10인가?`  
  - 어린 승객 그룹의 생존 패턴이 뚜렷하면 도움이 되는 질문입니다.
- 트리는 이런 후보를 비교해 다음 예측이 쉬워지는 질문을 고릅니다.

핵심은 숫자 계산이 아니라 **나눈 뒤에 각 그룹의 성격이 더 또렷해졌는지**입니다.


#### 자동으로 질문 고르기
- 사람이 `성별부터 볼까, 객실 등급부터 볼까?`를 하나하나 정하지 않습니다.  
- 모델이 여러 변수와 조건 후보를 시험해 봅니다.  
- 타이타닉 데이터에서는 `Sex`, `Pclass`, `Age`, `Fare`, `SibSp`, `Parch` 같은 후보를 비교합니다.  
- 그중 생존 여부를 더 잘 구분하게 해 주는 질문을 선택하고, 같은 과정을 반복합니다.

### 정리
- 결정 트리는 **좋은 질문을 차례로 고르는 모델**입니다.  
- 질문을 잘 고를수록 마지막 예측이 쉬워집니다.  
- 너무 많은 질문을 이어가면 훈련 데이터만 외울 수 있으므로 깊이를 제한합니다.


### 2.4 Scikit-learn으로 결정 트리 분류기 만들기


In [ ]:
from sklearn.datasets import load_breast_cancer  # 사이킷런 도구를 불러옵니다.
from sklearn.tree import DecisionTreeClassifier, plot_tree  # 결정트리 분류 모델, 트리 구조 시각화 함수입니다.
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.

# 데이터 불러오기
data = load_breast_cancer()
X, y = data.data, data.target

# 결정 트리 모델 학습
model = DecisionTreeClassifier(max_depth=3, random_state=42)  # 결정트리입니다. max_depth는 깊이, random_state는 재현용입니다.
model.fit(X, y)  # 데이터로 모델이나 변환 기준을 학습합니다.

# 트리 시각화
plt.figure(figsize=(12, 6))  # 그래프 크기와 도화지를 설정합니다.
plot_tree(model, feature_names=data.feature_names, class_names=data.target_names, filled=True, impurity=False)  # 결정트리 구조를 그림으로 표시합니다.
plt.show()  # 그래프를 화면에 출력합니다.

### 2.5 가지치기 (Pruning)
- 트리가 너무 깊어지면 훈련 데이터에 과적합됨  
- 해결 방법:  
  - `max_depth`: 트리의 최대 깊이 제한  
  - `min_samples_split`: 분할하기 위한 최소 샘플 수  
  - `min_samples_leaf`: 리프 노드에 필요한 최소 샘플 수  
  
  <img src="image/Prune.png">  

  이미지 출처 : https://developers.google.com/machine-learning/decision-forests/overfitting-and-pruning?hl=ko

### 체크포인트
- 결정 트리는 여러 질문 후보 중 예측하기 쉬운 그룹을 만드는 질문을 고른다.  
- 트리가 너무 깊으면 과적합되므로 가지치기로 제어해야 한다.  
- Scikit-learn을 사용하면 결정 트리 모델을 손쉽게 구현할 수 있다.  


## 3. 앙상블 기법 (Ensemble Methods)

### 3.1 앙상블 기법이란?
- 단일 모델 하나보다, 여러 개의 모델을 결합하면 더 좋은 성능을 낼 수 있습니다.  
- 이를 **앙상블(ensemble)** 기법이라고 합니다.  
- 아이디어: "여러 사람이 투표하면 더 정확한 답을 얻을 수 있다."  


### 3.2 배깅 (Bagging: Bootstrap Aggregating)
- 데이터 샘플을 여러 번 복원추출(bootstrap) 각각의 모델을 학습시킴  
- 모든 모델의 예측을 `평균(회귀)` 또는 `다수결(분류)`로 결합  
- 분산(Variance)을 줄이고, 과적합 위험을 완화  

👉 대표적인 배깅 기법: **랜덤 포레스트(Random Forest)**  


### 3.3 랜덤 포레스트 (Random Forest)
- 여러 개의 **결정 트리(Decision Tree)** 를 학습시킨 뒤,  
  예측을 모아 최종 결과를 결정하는 알고리즘  
- 각 트리는 데이터와 특성을 랜덤하게 선택 → 다양성을 확보  
- 장점:  
  - 단일 결정 트리보다 과적합 위험이 낮음  
  - 변수 중요도(Feature Importance)를 제공


### 3.4 Scikit-learn으로 랜덤 포레스트 실습


### 변수 중요도 보기
변수 중요도는 랜덤 포레스트가 예측할 때 **어떤 변수를 많이 참고했는지** 보여주는 값입니다.

타이타닉 예측이라면 `Sex`, `Pclass`, `Age` 중 어떤 정보가 생존 여부 판단에 더 크게 쓰였는지 보는 식입니다.  
숫자만 출력하면 감이 잘 오지 않으므로, 막대그래프로 상위 변수를 확인합니다.


In [ ]:
from sklearn.datasets import load_breast_cancer  # 사이킷런 도구를 불러옵니다.
from sklearn.ensemble import RandomForestClassifier  # 랜덤포레스트 분류 모델입니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.metrics import accuracy_score  # 정확도 지표 함수입니다.
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.

# 데이터 불러오기
data = load_breast_cancer()
X, y = data.data, data.target

# 학습/테스트 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.

# 랜덤 포레스트 모델 학습
rf = RandomForestClassifier(n_estimators=100, random_state=42)  # 랜덤포레스트입니다. n_estimators는 트리 수입니다.
rf.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.

# 예측 및 평가
y_pred = rf.predict(X_test)  # 학습한 모델로 새 값을 예측합니다.
print("정확도:", accuracy_score(y_test, y_pred))  # 정확도 비율을 계산합니다.

# 변수 중요도 시각화
feature_importances = pd.Series(rf.feature_importances_, index=data.feature_names)  # 변수별 중요도를 만듭니다.
top_importances = feature_importances.sort_values(ascending=False).head(10)  # 상위 10개 변수를 선택합니다.

plt.figure(figsize=(8, 5))  # 그래프 크기와 도화지를 설정합니다.
top_importances.sort_values().plot(kind="barh", color="#4C78A8")  # 막대그래프로 표시합니다.
plt.title("Random Forest Feature Importance Top 10")  # 그래프 제목을 설정합니다.
plt.xlabel("Importance")  # x축 이름을 설정합니다.
plt.ylabel("Feature")  # y축 이름을 설정합니다.
plt.tight_layout()  # 그래프 여백을 정리합니다.
plt.show()  # 그래프를 화면에 출력합니다.


### 3.5 랜덤 포레스트의 장점과 한계
- **장점**  
  - 단일 결정 트리보다 일반화 성능이 높음  
  - 다양한 데이터셋에서 안정적인 성능  
  - 변수 중요도를 통해 해석 가능  

- **한계**  
  - 단일 트리에 비해 학습 속도가 느릴 수 있음  
  - 매우 많은 트리를 사용할 경우 계산량 증가  
  - 최신 알고리즘에 비해 예측 성능이 떨어짐

### ✅ 체크포인트
- 배깅은 데이터를 여러 번 샘플링하여 여러 모델을 학습시키는 방법이다.  
- 랜덤 포레스트는 여러 결정 트리를 조합한 대표적인 배깅 기법이다.  
- 랜덤 포레스트는 과적합 위험을 줄이고, 안정적이고 강력한 성능을 제공한다.  
- 변수 중요도를 통해 어떤 변수가 중요한지 확인할 수 있다.  


## 4. 부스팅 (Boosting)

### 4.1 부스팅 개념
- 배깅과 달리, 여러 모델을 **순서대로 학습**시키는 방법  
- 앞 모델이 놓친 부분을 다음 모델이 이어서 보완  
- 작은 모델들을 차례로 붙여 전체 예측을 조금씩 개선  
- 대표 알고리즘: **Gradient Boosting, AdaBoost, XGBoost**

랜덤 포레스트처럼 데이터를 여러 묶음으로 따로 뽑는 방식이 아닙니다.  
같은 데이터를 보되, 앞에서 어려웠던 사례를 다음 모델이 더 유심히 보게 만듭니다.



### 4.2 Gradient Boosting
Gradient Boosting은 **초안을 여러 번 고치는 방식**과 비슷합니다.

1. 첫 모델이 대략적인 답안을 만듭니다.  
2. 다음 모델은 앞 답안에서 틀린 부분을 찾아 고칩니다.  
3. 그 다음 모델도 아직 남은 틀린 부분을 조금씩 고칩니다.  
4. 이 과정을 반복하면 전체 답안이 점점 정교해집니다.

`learning_rate`는 한 번에 고치는 폭입니다.
- 너무 크게 고치면 앞뒤가 흔들릴 수 있습니다.  
- 작게 고치면 안정적이지만 더 많은 트리가 필요합니다.

장점: 높은 정확도  
단점: 순서대로 고쳐가야 하므로 학습 속도가 느릴 수 있음


### 정리  

배깅(Bagging)은 여러 사람에게 같은 문제를 따로 풀게 한 뒤,  
답을 모아 평균이나 다수결로 결정하는 방식과 비슷합니다.  
 
부스팅(Boosting)은 한 사람이 만든 초안을 다음 사람이 고치고,  
그 다음 사람이 또 고치면서 답을 다듬는 방식과 비슷합니다.  

배깅은 **여러 답을 모아 안정적으로 결정**하고,  
부스팅은 **앞 단계의 부족한 부분을 차례로 보완**합니다.

In [ ]:
### Gradient Boosting 실습 (scikit-learn)

from sklearn.ensemble import GradientBoostingClassifier  # 그래디언트 부스팅 분류 모델입니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.metrics import accuracy_score  # 정확도 지표 함수입니다.
from sklearn.datasets import load_breast_cancer  # 사이킷런 도구를 불러옵니다.

# 데이터
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.

# 모델 학습
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)  # 부스팅 모델입니다. learning_rate는 보폭입니다.
gb.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.

# 예측 및 평가
y_pred = gb.predict(X_test)  # 학습한 모델로 새 값을 예측합니다.
print("Gradient Boosting 정확도:", accuracy_score(y_test, y_pred))  # 정확도 비율을 계산합니다.

### 4.3 XGBoost
- Gradient Boosting을 최적화한 라이브러리  
- 장점:  
  - 더 빠른 학습 속도  
  - 정규화(L1, L2) 지원 → 과적합 방지  
  - 대규모 데이터에서도 효율적  
- Kaggle 대회에서 자주 사용되는 강력한 알고리즘 


In [ ]:
### XGBoost 실습

import warnings
warnings.filterwarnings("ignore", category=UserWarning)  # XGBoost 실행 중 표시되는 UserWarning을 숨깁니다.
import xgboost as xgb  # XGBoost 모델 라이브러리입니다.
from xgboost import XGBClassifier  # XGBoost 분류 모델입니다.

# 모델 학습
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42, eval_metric="logloss")  # XGBoost 모델입니다. 트리 수와 학습률을 정합니다.
xgb_model.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.

# 예측 및 평가
y_pred = xgb_model.predict(X_test)  # 학습한 모델로 새 값을 예측합니다.
print("XGBoost 정확도:", accuracy_score(y_test, y_pred))  # 정확도 비율을 계산합니다.

### 4.4 Gradient Boosting vs XGBoost
- **Gradient Boosting**: 구현이 단순, 기본적인 부스팅 개념 학습에 적합  
- **XGBoost**: 속도와 성능이 뛰어나 실무 및 대규모 데이터셋에 적합  


### ✅ 체크포인트
- 부스팅은 앞 모델이 놓친 부분을 다음 모델이 보완하는 방식이다.  
- Gradient Boosting은 기본적인 부스팅 알고리즘이다.  
- XGBoost는 Gradient Boosting을 개선하여 속도와 성능을 강화한 알고리즘이다.  
- 학습률과 트리 개수는 부스팅 알고리즘에서 중요한 하이퍼파라미터이다.  


## 5. 모델 평가와 비교



### 5.1 왜 모델 평가가 중요한가?
- 단일 성능 지표만 보면 과적합을 놓치기 쉽습니다.  
- 훈련 데이터뿐 아니라 **검증 데이터, 교차 검증 결과**로 모델을 평가해야 합니다.  
- 특히 **결정 트리 vs 랜덤 포레스트 vs 부스팅(XGBoost)** 모델의 차이를 비교할 수 있어야 합니다.  



### 5.2 교차 검증 (Cross Validation)
- 데이터를 여러 폴드로 나눠 학습과 평가를 반복 → 평균 성능 확인  
- 안정적이고 일반화된 성능 평가 가능


In [ ]:
from sklearn.model_selection import cross_val_score  # 교차검증 점수 함수입니다.
from sklearn.tree import DecisionTreeClassifier  # 결정트리 분류 모델입니다.
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier  # 랜덤포레스트 분류 모델, 그래디언트 부스팅 분류 모델입니다.
import warnings
warnings.filterwarnings("ignore", category=UserWarning)  # XGBoost 실행 중 표시되는 UserWarning을 숨깁니다.
import xgboost as xgb  # XGBoost 모델 라이브러리입니다.
import numpy as np  # 배열과 수치 연산 라이브러리입니다.

# 모델 정의
dt = DecisionTreeClassifier(max_depth=3, random_state=42)  # 결정트리입니다. max_depth는 깊이, random_state는 재현용입니다.
rf = RandomForestClassifier(n_estimators=100, random_state=42)  # 랜덤포레스트입니다. n_estimators는 트리 수입니다.
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)  # 부스팅 모델입니다. learning_rate는 보폭입니다.
xgb_model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, eval_metric="logloss")  # XGBoost 모델입니다. 트리 수와 학습률을 정합니다.

models = {"Decision Tree": dt, "Random Forest": rf, "Gradient Boosting": gb, "XGBoost": xgb_model}

# 교차 검증
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5)  # 교차검증 점수를 계산합니다.
    print(f"{name} 평균 정확도: {np.mean(scores):.4f}")  # 평균을 계산합니다.

### 5.3 과적합 확인
- **결정 트리**: 깊이가 깊어질수록 과적합 위험 ↑  
- **랜덤 포레스트**: 과적합 방지 효과 ↑  
- **부스팅(XGBoost)**: 높은 성능, 그러나 과적합 가능 → 학습률과 트리 수 조정 필요  

👉 해결책:  
- 교차 검증으로 안정적 평가  
- `max_depth`, `n_estimators`, `learning_rate` 등 하이퍼파라미터 조정  



### 5.4 성능 비교 예시 출력
- 보통 결과는 다음과 같이 나올 수 있음 (데이터셋에 따라 다름)  

| 모델               | 평균 정확도 |
|--|-|
| Decision Tree      | 0.90        |
| Random Forest      | 0.95        |
| Gradient Boosting  | 0.96        |
| XGBoost            | 0.97        |



### ✅ 체크포인트
- 모델은 반드시 교차 검증으로 평가해야 한다.  
- 결정 트리는 과적합되기 쉬운 반면, 랜덤 포레스트와 부스팅은 일반화 성능이 더 높다.  
- XGBoost는 대규모 데이터셋에서 뛰어난 성능을 보인다.  
- 하이퍼파라미터 튜닝을 통해 성능을 추가로 개선할 수 있다.  


### 결정 트리와 앙상블 기법 – 실습 문제

### 문제 1. 결정 트리 학습 및 시각화
Breast Cancer 데이터셋을 불러와서,  
`max_depth=3`인 결정 트리를 학습하고 시각화하세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 먼저 필요한 값을 계산합니다.
# 계산 결과를 표나 그래프로 확인합니다.

from sklearn.datasets import load_breast_cancer  # 실습용 유방암 데이터를 불러오는 함수입니다.
from sklearn.tree import DecisionTreeClassifier, plot_tree  # 결정 트리 모델과 트리 시각화 함수입니다.
import matplotlib.pyplot as plt  # 그래프를 그리기 위한 Matplotlib입니다.

data = load_breast_cancer()  # 예제 데이터셋을 불러옵니다.
X, y = data.data, data.target  # 모델에 넣을 입력 데이터입니다.

model = DecisionTreeClassifier(max_depth=3, random_state=42)  # 결정트리입니다. max_depth는 깊이, random_state는 재현용입니다.
model.fit(X, y)  # 데이터로 모델이나 변환 기준을 학습합니다.

plt.figure(figsize=(12,6))  # 그래프 크기와 도화지를 설정합니다.
plot_tree(model, feature_names=data.feature_names, class_names=data.target_names, filled=True, impurity=False)  # 결정트리 구조를 그림으로 표시합니다.
plt.show()  # 그래프를 화면에 표시합니다.

```
</details>



In [ ]:
# 여기에 정답을 작성하세요
from sklearn.datasets import load_breast_cancer  # 사이킷런 도구를 불러옵니다.
from sklearn.tree import DecisionTreeClassifier, plot_tree  # 결정트리 분류 모델, 트리 구조 시각화 함수입니다.
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.

data = load_breast_cancer()
X, y = data.data, data.target

### 문제 2. 랜덤 포레스트 학습 및 정확도 평가
Breast Cancer 데이터셋을 사용하여  
`n_estimators=100`인 랜덤 포레스트 분류기를 학습하고 테스트 정확도를 구하세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 필요한 값을 만들고 결과를 확인합니다.
# 문제에서 요구한 결과를 계산하고 확인합니다.

from sklearn.ensemble import RandomForestClassifier  # 여러 결정 트리를 묶는 랜덤 포레스트 모델입니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터를 나누는 함수입니다.
from sklearn.metrics import accuracy_score  # 정확도를 계산하는 함수입니다.

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.

rf = RandomForestClassifier(n_estimators=100, random_state=42)  # 랜덤포레스트입니다. n_estimators는 트리 수입니다.
rf.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.

y_pred = rf.predict(X_test)  # 학습한 모델로 새 값을 예측합니다.
print("정확도:", accuracy_score(y_test, y_pred))  # 결과를 화면에 출력합니다.

```
</details>

In [ ]:
# 여기에 정답을 작성하세요
from sklearn.datasets import load_breast_cancer  # 사이킷런 도구를 불러옵니다.
from sklearn.tree import DecisionTreeClassifier, plot_tree  # 결정트리 분류 모델, 트리 구조 시각화 함수입니다.
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.

data = load_breast_cancer()
X, y = data.data, data.target

### 문제 3. 변수 중요도 시각화
문제 2에서 학습한 랜덤 포레스트 모델의 **상위 5개 변수 중요도**를 막대그래프로 시각화하세요.


변수 중요도란?
- 모델이 예측할 때 각 입력 변수가 얼마나 많이 도움이 되었는지 나타내는 값입니다.  
- 값이 높을수록 그 변수가 결과를 판단하는 데 더 많이 쓰였다는 뜻입니다.  
- 랜덤 포레스트에서는 여러 트리가 반복해서 참고한 변수가 중요하게 나타납니다.  
- `feature_importances_` 값을 그래프로 그리면 어떤 변수가 눈에 띄는지 빠르게 볼 수 있습니다.

<details>
<summary>정답 보기</summary>

```python
import pandas as pd  # 표 데이터를 다루기 위한 Pandas입니다.
import matplotlib.pyplot as plt  # 그래프를 그리기 위한 Matplotlib입니다.

feature_importances = pd.Series(rf.feature_importances_, index=data.feature_names)  # 변수별 중요도를 만듭니다.
top5 = feature_importances.sort_values(ascending=False).head(5)  # 상위 5개 변수를 선택합니다.

plt.figure(figsize=(8, 4))  # 그래프 크기와 도화지를 설정합니다.
top5.sort_values().plot(kind="barh", color="#4C78A8")  # 막대그래프로 표시합니다.
plt.title("Random Forest Feature Importance Top 5")  # 그래프 제목을 설정합니다.
plt.xlabel("Importance")  # x축 이름을 설정합니다.
plt.ylabel("Feature")  # y축 이름을 설정합니다.
plt.tight_layout()  # 그래프 여백을 정리합니다.
plt.show()  # 그래프를 화면에 표시합니다.
```
</details>


In [ ]:
# 실행만 하시면 됩니다
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.

feature_importances = pd.Series(rf.feature_importances_, index=data.feature_names)  # 변수별 중요도를 만듭니다.
top5 = feature_importances.sort_values(ascending=False).head(5)  # 상위 5개 변수를 선택합니다.

plt.figure(figsize=(8, 4))  # 그래프 크기와 도화지를 설정합니다.
top5.sort_values().plot(kind="barh", color="#4C78A8")  # 막대그래프로 표시합니다.
plt.title("Random Forest Feature Importance Top 5")  # 그래프 제목을 설정합니다.
plt.xlabel("Importance")  # x축 이름을 설정합니다.
plt.ylabel("Feature")  # y축 이름을 설정합니다.
plt.tight_layout()  # 그래프 여백을 정리합니다.
plt.show()  # 그래프를 화면에 출력합니다.

### 문제 4. Gradient Boosting 적용
Breast Cancer 데이터셋에서 `learning_rate=0.1`, `n_estimators=100`인  
GradientBoostingClassifier를 학습하고 정확도를 출력하세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 필요한 값을 만들고 결과를 확인합니다.
# 문제에서 요구한 결과를 계산하고 확인합니다.

from sklearn.ensemble import GradientBoostingClassifier  # 부스팅 방식의 앙상블 모델입니다.

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)  # 부스팅 모델입니다. learning_rate는 보폭입니다.
gb.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.

y_pred = gb.predict(X_test)  # 학습한 모델로 새 값을 예측합니다.
print("Gradient Boosting 정확도:", accuracy_score(y_test, y_pred))  # 결과를 화면에 출력합니다.

```
</details>



In [ ]:
# 여기에 정답을 작성하세요
from sklearn.datasets import load_breast_cancer  # 사이킷런 도구를 불러옵니다.
from sklearn.tree import plot_tree  # 트리 구조 시각화 함수입니다.
from sklearn.ensemble import GradientBoostingClassifier  # 그래디언트 부스팅 분류 모델입니다.
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.

### 문제 5. XGBoost 적용 및 비교
Breast Cancer 데이터셋에서 XGBoost 모델을 학습하고,  
Gradient Boosting과 정확도를 비교하세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 필요한 값을 만들고 결과를 확인합니다.
# 문제에서 요구한 결과를 계산하고 확인합니다.

import warnings
warnings.filterwarnings("ignore", category=UserWarning)  # XGBoost 실행 중 표시되는 UserWarning을 숨깁니다.
from xgboost import XGBClassifier  # XGBoost 분류 모델입니다.

xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3,  # XGBoost 모델입니다. 트리 수와 학습률을 정합니다.
                          random_state=42, eval_metric="logloss")  # 재현성과 평가 기준 설정입니다.
xgb_model.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.

y_pred = xgb_model.predict(X_test)  # 학습한 모델로 새 값을 예측합니다.
print("XGBoost 정확도:", accuracy_score(y_test, y_pred))  # 결과를 화면에 출력합니다.

```
</details>

In [ ]:
# 여기에 정답을 작성하세요
from sklearn.datasets import load_breast_cancer  # 사이킷런 도구를 불러옵니다.
import warnings
warnings.filterwarnings("ignore", category=UserWarning)  # XGBoost 실행 중 표시되는 UserWarning을 숨깁니다.
from xgboost import XGBClassifier  # XGBoost 분류 모델입니다.
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.

data = load_breast_cancer()
X, y = data.data, data.target

### 문제 6. 교차 검증으로 모델 비교
결정 트리, 랜덤 포레스트, Gradient Boosting, XGBoost 모델을  
5겹 교차 검증하여 평균 정확도를 각각 출력하세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.model_selection import cross_val_score  # 교차 검증 점수를 계산하는 함수입니다.
import numpy as np  # 배열 계산을 위한 Numpy입니다.

models = {  # 비교할 모델들을 딕셔너리로 묶습니다.
    "Decision Tree": DecisionTreeClassifier(max_depth=3, random_state=42),  # 결정트리입니다. max_depth는 깊이, random_state는 재현용입니다.
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),  # 랜덤포레스트입니다. n_estimators는 트리 수입니다.
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),  # 부스팅 모델입니다. learning_rate는 보폭입니다.
    "XGBoost": XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42,  # XGBoost 모델입니다. 트리 수와 학습률을 정합니다.
                              eval_metric="logloss")  # 평가 기준을 로그손실로 설정합니다.
}

for name, model in models.items():  # 값을 하나씩 꺼내 반복합니다.
    scores = cross_val_score(model, X, y, cv=5)  # 이름과 점수를 저장할 딕셔너리입니다.
    print(f"{name} 평균 정확도: {np.mean(scores):.4f}")  # 결과를 화면에 출력합니다.

```
</details>



In [ ]:
# 여기에 정답을 작성하세요
from sklearn.datasets import load_breast_cancer  # 사이킷런 도구를 불러옵니다.
import warnings
warnings.filterwarnings("ignore", category=UserWarning)  # XGBoost 실행 중 표시되는 UserWarning을 숨깁니다.
from xgboost import XGBClassifier  # XGBoost 분류 모델입니다.
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.

data = load_breast_cancer()
X, y = data.data, data.target

### ✅ 체크포인트
- 결정 트리는 이해하기 쉽지만 과적합 위험이 크다.  
- 랜덤 포레스트는 배깅 기법으로 안정적 성능을 낸다.  
- Gradient Boosting은 앞 모델이 놓친 부분을 다음 모델이 보완하는 강력한 기법이다.  
- XGBoost는 최적화된 부스팅 알고리즘으로, 속도와 성능이 뛰어나다.  
- 교차 검증을 통해 모델의 일반화 성능을 안정적으로 비교할 수 있다.  


# 차원 축소 (Dimensionality Reduction)

## 1. 개요

### 배경
현실 데이터는 수십, 수백 개의 특성을 가진 **고차원 데이터**인 경우가 많습니다.  
차원이 높아지면 → 시각화가 어려워지고, 계산량이 폭발하며, 모델이 과적합되는 문제가 발생합니다.  

👉 **차원 축소(Dimensionality Reduction)** 는  
- 데이터의 중요한 정보를 유지하면서,  
- 불필요하거나 중복된 차원을 줄여  
- 시각화, 처리 효율성, 모델 성능 향상에 기여합니다.

## 학습 목표

- 차원 축소가 필요한 상황을 설명할 수 있다.  
- PCA와 LDA를 비유로 이해하고 실제 데이터에 적용할 수 있다.  
- 차원 축소 전후의 시각화와 모델 성능을 비교할 수 있다.


## 주요 학습 개념

- **차원 축소란?**
  - 특성이 너무 많을 때 핵심 정보만 남겨 보기 쉽게 만드는 방법  
  - 시각화, 처리 속도, 모델 비교에 활용

- **PCA (Principal Component Analysis)**  
  - 여러 특성을 더 적은 개수의 요약 특성으로 바꾸는 방법  
  - 데이터 전체 모양을 잘 보여주는 방향으로 다시 정리  
  - scikit-learn으로 적용하고 시각화

- **LDA (Linear Discriminant Analysis)**  
  - 정답 라벨을 활용해 클래스가 더 잘 보이도록 정리하는 방법  
  - scikit-learn으로 적용하고 시각화

- **차원 축소 활용**  
  - 2D 그래프로 데이터 구조 확인  
  - 차원 축소 전후 모델 성능 비교  
  - 피처 선택과 차원 축소 차이 구분


## 2. 들어가기

데이터가 가진 **차원(dimension)** 이란, 하나의 데이터가 가지고 있는 **특성(feature)** 의 개수를 의미합니다.  
예:  
- 키, 몸무게, 나이 → 3차원 데이터  
- 픽셀 784개로 이루어진 MNIST 숫자 이미지 → 784차원 데이터


👉 차원이 커질수록 나타나는 문제:  
- **보기 어려움**: 특성이 4개만 넘어도 한 번에 그림으로 보기 어렵다.  
- **처리 부담 증가**: 특성이 많을수록 모델이 확인해야 할 정보가 늘어난다.  
- **패턴 파악 어려움**: 중요한 신호보다 잡음이 더 눈에 띌 수 있다.  

  <img src="image/dimensionality.png">  

이미지 출처 : https://www.visiondummy.com/2014/04/curse-dimensionality-affect-classification/#google_vignette

따라서 차원을 줄이는 기술, 즉 **차원 축소(Dimensionality Reduction)** 가 필요하다.  


## 3. 차원 축소 기본 개념

### 3.1 차원 축소란?
- 원래 데이터의 중요한 정보를 유지하면서 **특성의 개수를 줄이는 과정**  
- 불필요한 변수 제거, 데이터 압축, 시각화에 활용  

<img src="image/Dimensionality_Reduction.jpg">  

이미지 출처 : https://blog.roboflow.com/what-is-dimensionality-reduction/


### 3.2 차원 축소의 이점
1. **시각화 용이**  
   - 2D, 3D로 축소하여 데이터 패턴을 그림으로 확인 가능  
2. **처리 효율성 향상**  
   - 계산량이 줄어 모델 훈련 속도 증가  
3. **잡음 제거**  
   - 중요하지 않은 특성을 줄여 모델 성능 향상  
4. **과적합 방지**  
   - 복잡성을 줄여 일반화 성능 향상


### 3.3 차원 축소 방법
- **PCA (주성분 분석)**: 데이터 전체 모양이 잘 보이도록 여러 특성을 적은 수의 요약 특성으로 바꿈  
- **LDA (선형 판별 분석)**: 정답 라벨을 참고해 클래스가 더 잘 구분되도록 특성을 줄임  
- (심화) t-SNE, UMAP: 복잡한 데이터의 모양을 그림으로 확인할 때 자주 사용


### 3.4 간단한 예시 (Iris 데이터셋 시각화 전후)


In [ ]:
from sklearn.datasets import load_iris  # 사이킷런 도구를 불러옵니다.
from sklearn.decomposition import PCA  # 차원축소 도구입니다.
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.
import seaborn as sns  # 통계 그래프 시각화 도구입니다.
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)  # FutureWarning을 숨깁니다.


# 데이터 불러오기
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)  # 데이터프레임을 직접 만듭니다.
df["target"] = iris.target

# ---------------------
# 전(before): 4D → pairplot
# ---------------------
sns.pairplot(df, hue="target", vars=iris.feature_names)  # 시본 그래프를 그립니다.
plt.suptitle("Before Dimensionality Reduction (4D Feature Space)", y=1.02)
plt.show()  # 그래프를 화면에 출력합니다.

# ---------------------
# 후(after): PCA → 2D
# ---------------------
pca = PCA(n_components=2)  # PCA 차원축소입니다. n_components는 남길 요약 특성 수입니다.
pca_result = pca.fit_transform(iris.data)  # PCA를 적용해 데이터를 2차원으로 바꿉니다.

df_pca = pd.DataFrame(pca_result, columns=["PC1", "PC2"])  # 데이터프레임을 직접 만듭니다.
df_pca["target"] = iris.target

plt.figure(figsize=(6,5))  # 그래프 크기와 도화지를 설정합니다.
sns.scatterplot(x="PC1", y="PC2", hue="target", data=df_pca, palette="Set1")  # 시본 그래프를 그립니다.
plt.title("After Dimensionality Reduction (PCA 2D)")  # 그래프 제목을 설정합니다.
plt.show()  # 그래프를 화면에 출력합니다.


### 체크포인트
- 차원은 데이터가 가진 특성 개수다.  
- 특성이 많으면 보기 어렵고, 모델이 확인할 정보도 늘어난다.  
- 차원 축소는 시각화, 처리 속도, 성능 비교에 도움을 준다.  
- 대표 기법으로 PCA와 LDA가 있다.  


## 4. PCA (Principal Component Analysis)

### 4.1 PCA란?
PCA는 특성이 많은 데이터를 **더 적은 개수의 요약 특성**으로 바꾸는 방법입니다.

예를 들어 `키(cm)`와 `키(m)`는 단위만 다를 뿐 거의 같은 정보를 담고 있습니다. 둘 다 따로 들고 가기보다 하나로 정리할 수 있습니다.

PCA도 비슷하게 서로 함께 움직이는 특성들을 섞어, 데이터의 큰 흐름을 설명하는 새 축(`PC1`, `PC2`)으로 압축합니다.


### 4.2 PCA 흐름

흐름은 다음 정도로 보면 됩니다.

1. 특성들의 단위를 맞춥니다.  
2. 함께 변하는 특성들을 찾아 하나의 요약 방향으로 묶습니다.  
3. 데이터 흐름을 가장 잘 설명하는 축부터 남깁니다.  
4. 필요한 개수만 선택합니다.  
   - 4개 특성을 2개 요약 특성으로 줄이면 2D 그래프로 볼 수 있습니다.


#### 예: Iris 데이터
- 원래 특성: 꽃받침 길이, 꽃받침 폭, 꽃잎 길이, 꽃잎 폭  
- PCA 적용 후: `PC1`, `PC2`라는 두 개의 요약 특성  

`PC1`, `PC2`는 원래 열 하나를 그대로 복사한 값이 아니라, 여러 열의 정보를 섞어서 만든 **새로운 요약 좌표**입니다.


#### PCA 결과를 볼 때

PCA를 적용한 뒤에는 숫자보다 그림을 먼저 보는 것이 좋습니다.

- 점들이 색깔별로 어느 정도 나뉘어 보이는가?  
- 서로 다른 클래스가 겹치는 구간은 어디인가?  
- 원본보다 2D 그래프에서 구조가 더 잘 보이는가?

PCA는 정답 라벨을 보고 축을 만드는 방법은 아닙니다.  
그래서 클래스가 항상 깔끔하게 나뉘지는 않습니다. 대신 데이터 전체의 큰 모양을 빠르게 살펴보는 데 유용합니다.


#### 단위 맞추기

PCA를 적용하기 전에는 특성들의 단위를 맞추는 것이 좋습니다.

예를 들어 한 열은 `cm`, 다른 열은 `원` 단위라면 숫자 크기가 크게 다릅니다.  
이 상태로 바로 비교하면 큰 숫자를 가진 열이 더 중요해 보일 수 있습니다.

`StandardScaler`는 각 열을 비슷한 출발선에 세워 주는 전처리 도구입니다.


### 4.3 PCA 시각적 직관

산점도에서 점들이 대각선 방향으로 길게 퍼져 있다면, PCA는 그 방향을 첫 번째 축(`PC1`)으로 잡습니다.  
그 다음에는 `PC1`로 설명되지 않는 남은 변화를 두 번째 축(`PC2`)으로 잡습니다.

즉, PCA는 원래 특성을 단순히 버리는 것이 아니라 **데이터가 많이 변하는 방향을 새 좌표축으로 다시 잡는 방법**입니다.


### 4.4 Scikit-learn으로 PCA 구현


In [ ]:
from sklearn.decomposition import PCA  # PCA 차원 축소 도구입니다.
from sklearn.preprocessing import StandardScaler  # 특성 스케일을 맞추는 도구입니다.
from sklearn.datasets import load_iris  # 사이킷런 도구를 불러옵니다.
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.

# 데이터 불러오기
iris = load_iris()
X = iris.data
y = iris.target

# 1) 특성 스케일 맞추기
X_scaled = StandardScaler().fit_transform(X)  # 서로 다른 단위를 비슷한 기준으로 맞춥니다.

# 2) PCA로 2개의 요약 특성 만들기
pca = PCA(n_components=2)  # n_components는 남길 요약 특성 수입니다.
X_pca = pca.fit_transform(X_scaled)  # PCA를 적용해 데이터를 2차원으로 바꿉니다.

# 3) DataFrame 변환
df_pca = pd.DataFrame(X_pca, columns=["PC1", "PC2"])  # 시각화를 위해 표 형태로 바꿉니다.
df_pca["target"] = y

# 4) 시각화
plt.figure(figsize=(8,6))  # 그래프 크기와 도화지를 설정합니다.
for target in set(y):
    subset = df_pca[df_pca["target"] == target]
    plt.scatter(subset["PC1"], subset["PC2"], label=iris.target_names[target])  # 산점도를 그립니다.
plt.xlabel("PC1")  # x축 이름을 설정합니다.
plt.ylabel("PC2")  # y축 이름을 설정합니다.
plt.legend()  # 범례를 표시합니다.
plt.title("PCA on Iris Dataset")  # 그래프 제목을 설정합니다.
plt.show()  # 그래프를 화면에 출력합니다.


### 4.5 PCA 결과 확인
PCA를 적용하면 데이터 모양이 바뀝니다.  
먼저 원본 특성 수와 PCA 적용 후 특성 수를 비교해 봅니다.

```python
print("원본 데이터 크기:", X.shape)
print("PCA 적용 후 크기:", X_pca.shape)
```

원본 Iris 데이터는 특성이 4개이고, 여기서는 PCA로 2개 요약 특성만 남겼습니다.  
이렇게 줄이면 산점도로 데이터의 전체 모양을 확인하기 쉬워집니다.


### 체크포인트
- PCA는 많은 특성을 적은 수의 요약 특성으로 바꾼다.  
- PCA를 적용하기 전에는 특성 스케일을 맞추는 것이 좋다.  
- PCA는 데이터 전체 모양을 빠르게 확인할 때 유용하다.  
- Scikit-learn으로 손쉽게 PCA를 적용할 수 있다.  


## 5. LDA (Linear Discriminant Analysis)

### 5.1 LDA란?
LDA는 **정답 라벨을 활용하는 차원 축소 방법**입니다.  
PCA가 전체 데이터의 큰 흐름을 기준으로 축을 잡는다면, LDA는 클래스가 서로 잘 갈라지는 방향을 기준으로 축을 잡습니다.

비유하면, 이름표가 붙은 학생들을 운동장에 다시 배치할 때 **같은 반 학생은 가까이**, **다른 반 학생은 멀리** 보이게 기준선을 잡는 것과 비슷합니다.


### 5.2 LDA 흐름
1. 각 데이터가 어떤 클래스에 속하는지 확인합니다.  
2. 같은 클래스 안의 흩어짐은 줄입니다.  
3. 서로 다른 클래스 사이의 거리는 키웁니다.  
4. 그 방향으로 차원을 줄여 2D 그래프나 분류 모델에 사용합니다.


### 5.3 LDA와 PCA 차이점
- PCA: 정답 라벨 없이 데이터 전체 모양을 보기 좋게 정리  
- LDA: 정답 라벨을 활용해 클래스가 더 잘 보이도록 정리  

PCA는 **전체 데이터가 가장 많이 퍼진 방향**을 찾고, LDA는 **클래스 사이가 가장 잘 벌어지는 방향**을 찾습니다.


### 5.4 Scikit-learn으로 LDA 구현


In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA  # LDA 차원축소/분류 도구입니다.
from sklearn.datasets import load_iris  # 사이킷런 도구를 불러옵니다.
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.

# 데이터 불러오기
iris = load_iris()
X = iris.data
y = iris.target

# LDA (2차원으로 축소)
lda = LDA(n_components=2)  # LDA입니다. 라벨 정보를 활용해 차원을 줄입니다.
X_lda = lda.fit_transform(X, y)  # 기준 학습과 변환을 한 번에 수행합니다.

# DataFrame 변환
df_lda = pd.DataFrame(X_lda, columns=["LD1", "LD2"])  # 데이터프레임을 직접 만듭니다.
df_lda["target"] = y

# 시각화
plt.figure(figsize=(8,6))  # 그래프 크기와 도화지를 설정합니다.
for target in set(y):
    subset = df_lda[df_lda["target"] == target]
    plt.scatter(subset["LD1"], subset["LD2"], label=iris.target_names[target])  # 산점도를 그립니다.
plt.xlabel("LD1")  # x축 이름을 설정합니다.
plt.ylabel("LD2")  # y축 이름을 설정합니다.
plt.legend()  # 범례를 표시합니다.
plt.title("LDA on Iris Dataset")  # 그래프 제목을 설정합니다.
plt.show()  # 그래프를 화면에 출력합니다.

### 5.5 LDA의 활용
- **시각화**: 고차원 데이터를 2D/3D로 줄여 시각적으로 클래스 분포 확인  
- **분류 성능 향상**: 차원을 줄이면서도 클래스 정보를 반영하여 더 잘 분리된 데이터 생성  
- **전처리 단계**: 분류 모델 적용 전 데이터 축소


### 체크포인트
- LDA는 클래스 정보를 활용하는 차원 축소 방법이다.  
- LDA는 서로 다른 클래스가 그래프에서 더 잘 보이도록 돕는다.  
- PCA와 달리 LDA는 정답 라벨을 함께 사용한다.  
- Scikit-learn을 통해 손쉽게 LDA를 적용하고 시각화할 수 있다.  


## 6. 차원 축소 활용

### 6.1 데이터 시각화
차원 축소의 가장 직관적인 효과는 **고차원 데이터를 2D 또는 3D로 시각화**할 수 있다는 점입니다.  

예: Iris 데이터셋을 PCA와 LDA로 축소 후 시각화


In [ ]:
# 이미 PCA, LDA로 변환된 데이터 (X_pca, X_lda)
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.

# PCA 결과 시각화
plt.figure(figsize=(8,6))  # 그래프 크기와 도화지를 설정합니다.
plt.scatter(X_pca[:,0], X_pca[:,1], c=y, cmap="viridis", edgecolor="k")  # 산점도를 그립니다.
plt.xlabel("PC1")  # x축 이름을 설정합니다.
plt.ylabel("PC2")  # y축 이름을 설정합니다.
plt.title("PCA Visualization (Iris)")  # 그래프 제목을 설정합니다.
plt.colorbar()
plt.show()  # 그래프를 화면에 출력합니다.

# LDA 결과 시각화
plt.figure(figsize=(8,6))  # 그래프 크기와 도화지를 설정합니다.
plt.scatter(X_lda[:,0], X_lda[:,1], c=y, cmap="rainbow", edgecolor="k")  # 산점도를 그립니다.
plt.xlabel("LD1")  # x축 이름을 설정합니다.
plt.ylabel("LD2")  # y축 이름을 설정합니다.
plt.title("LDA Visualization (Iris)")  # 그래프 제목을 설정합니다.
plt.colorbar()
plt.show()  # 그래프를 화면에 출력합니다.

👉 결과:  
- PCA는 데이터 전체 구조를 빠르게 파악하는 데 유용  
- LDA는 클래스별 그룹이 더 명확히 보이는지 확인하는 데 유용


### 6.2 모델 성능 비교
차원 축소 후 **모델을 학습**하여 성능 차이를 비교할 수 있습니다.  


In [ ]:
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 분류 모델입니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.metrics import accuracy_score  # 정확도 지표 함수입니다.

# 데이터 분할 (원본 데이터 vs PCA 변환 데이터)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.

# 원본 데이터
lr = LogisticRegression(max_iter=200)  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
lr.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.
y_pred = lr.predict(X_test)  # 학습한 모델로 새 값을 예측합니다.
print("원본 데이터 정확도:", accuracy_score(y_test, y_pred))  # 정확도 비율을 계산합니다.

# PCA 데이터
X_pca_train, X_pca_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.
lr.fit(X_pca_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.
y_pred_pca = lr.predict(X_pca_test)  # 학습한 모델로 새 값을 예측합니다.
print("PCA 데이터 정확도:", accuracy_score(y_test, y_pred_pca))  # 정확도 비율을 계산합니다.

# LDA 데이터
X_lda_train, X_lda_test, y_train, y_test = train_test_split(X_lda, y, test_size=0.2, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.
lr.fit(X_lda_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.
y_pred_lda = lr.predict(X_lda_test)  # 학습한 모델로 새 값을 예측합니다.
print("LDA 데이터 정확도:", accuracy_score(y_test, y_pred_lda))  # 정확도 비율을 계산합니다.

👉 보통:  
- PCA → 특성을 줄인 뒤에도 성능이 유지되는지 확인  
- LDA → 클래스 구분이 더 쉬워지는지 확인

### 6.3 피처 선택 vs 차원 축소
- **피처 선택 (Feature Selection)**  
  - 원래의 변수 중 중요한 것만 고름  
  - 해석이 쉽다 (예: “꽃받침 길이와 꽃잎 폭이 중요하다”)  

- **차원 축소 (Dimensionality Reduction)**  
  - 원래 변수를 그대로 고르는 것이 아니라, 여러 변수를 섞어 새로운 요약 특성을 만듦  
  - 해석은 어렵지만 데이터 구조를 간단히 볼 수 있다  

즉, 피처 선택은 "원래 특성 중 일부만 고르기", 차원 축소는 "새로운 요약 특성 만들기"입니다.


### 체크포인트
- 차원 축소는 데이터 시각화, 처리 속도, 모델 비교에 유용하다.  
- PCA는 데이터 전체 모양을 보기 좋게 정리한다.  
- LDA는 클래스 구분이 잘 보이도록 정리한다.  
- 차원 축소 후에도 모델 성능을 반드시 다시 평가해야 한다.  
- 피처 선택과 차원 축소는 다른 접근 방식이다.


### 차원 축소 – 실습 문제

### 문제 1. PCA 변환
Iris 데이터셋을 불러와서,  
`n_components=2`인 PCA를 적용한 뒤 2차원 산점도로 시각화하세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 데이터를 불러오고, 특성 스케일을 맞춘 뒤 PCA로 2차원 그래프를 그립니다.

from sklearn.decomposition import PCA  # PCA 차원 축소 도구입니다.
from sklearn.preprocessing import StandardScaler  # 특성 스케일을 맞추는 도구입니다.
from sklearn.datasets import load_iris  # 실습용 iris 데이터를 불러오는 함수입니다.
import matplotlib.pyplot as plt  # 그래프를 그리기 위한 Matplotlib입니다.

iris = load_iris()  # 예제 데이터셋을 불러옵니다.
X, y = iris.data, iris.target  # 모델에 넣을 입력 데이터입니다.

# 특성 스케일 맞추기
X_scaled = StandardScaler().fit_transform(X)  # 서로 다른 단위를 비슷한 기준으로 맞춥니다.

# PCA 적용
pca = PCA(n_components=2)  # PCA 차원축소입니다. n_components는 남길 요약 특성 수입니다.
X_pca = pca.fit_transform(X_scaled)  # PCA를 적용해 데이터를 2차원으로 바꿉니다.

# 시각화
plt.scatter(X_pca[:,0], X_pca[:,1], c=y, cmap="viridis", edgecolor="k")  # 산점도를 그립니다.
plt.xlabel("PC1")  # x축 이름을 설정합니다.
plt.ylabel("PC2")  # y축 이름을 설정합니다.
plt.title("PCA on Iris")  # 그래프 제목을 설정합니다.
plt.show()  # 그래프를 화면에 표시합니다.

```
</details>



In [ ]:
# 여기에 코드를 작성하세요 
from sklearn.decomposition import PCA  # 차원축소 도구입니다.
from sklearn.preprocessing import StandardScaler  # 특성 스케일을 맞추는 도구입니다.
from sklearn.datasets import load_iris  # 사이킷런 도구를 불러옵니다.
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.

# 데이터 불러오기
iris = load_iris()
X = iris.data
y = iris.target

### 문제 2. PCA 결과 크기 확인
문제 1에서 만든 `X_pca`의 크기를 확인하세요.  
원본 데이터와 비교하면 특성이 몇 개로 줄었는지 바로 볼 수 있습니다.

<details>
<summary>정답 보기</summary>

```python
print("원본 데이터 크기:", X.shape)  # 원본 데이터의 행과 열 개수를 확인합니다.
print("PCA 적용 후 크기:", X_pca.shape)  # PCA 적용 후 행과 열 개수를 확인합니다.
```
</details>


In [ ]:
print("원본 데이터 크기:", X.shape)  # 원본 데이터의 행과 열 개수를 확인합니다.
print("PCA 적용 후 크기:", X_pca.shape)  # PCA 적용 후 행과 열 개수를 확인합니다.


### 문제 3. LDA 변환
Iris 데이터셋에 LDA를 적용해 2차원으로 축소하고,  
클래스별로 다른 색으로 시각화하세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 필요한 값을 만들고 결과를 확인합니다.
# 문제에서 요구한 결과를 계산하고 확인합니다.

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA  # LDA 차원축소/분류 도구입니다.
from sklearn.datasets import load_iris  # 실습용 iris 데이터를 불러오는 함수입니다.
import pandas as pd  # 표 데이터를 다루기 위한 Pandas입니다.
import matplotlib.pyplot as plt  # 그래프를 그리기 위한 Matplotlib입니다.

# 데이터 불러오기
iris = load_iris()  # 예제 데이터셋을 불러옵니다.
X = iris.data  # 모델에 넣을 입력 데이터입니다.
y = iris.target  # 정답 값 또는 두 번째 배열을 준비합니다.

lda = LDA(n_components=2)  # LDA입니다. 라벨 정보를 활용해 차원을 줄입니다.
X_lda = lda.fit_transform(X, y)  # 학습과 변환을 한 번에 수행합니다.

plt.scatter(X_lda[:,0], X_lda[:,1], c=y, cmap="rainbow", edgecolor="k")  # 산점도를 그립니다.
plt.xlabel("LD1")  # x축 이름을 설정합니다.
plt.ylabel("LD2")  # y축 이름을 설정합니다.
plt.title("LDA on Iris")  # 그래프 제목을 설정합니다.
plt.show()  # 그래프를 화면에 표시합니다.

```
</details>

In [ ]:
# 여기에 코드를 작성하세요 
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA  # LDA 차원축소/분류 도구입니다.
from sklearn.datasets import load_iris  # 사이킷런 도구를 불러옵니다.
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.

# 데이터 불러오기
iris = load_iris()
X = iris.data
y = iris.target

### 문제 4. 차원 축소 후 분류 모델 성능 비교
Iris 데이터셋에서 **로지스틱 회귀** 모델을 적용했을 때,  
- 원본 데이터,  
- PCA 변환 데이터,  
- LDA 변환 데이터  

의 정확도를 비교하세요.  

<details>
<summary>정답 보기</summary>

```python
# 풀이 흐름: 입력 데이터와 정답 데이터를 준비합니다.
# 모델이나 변환기를 적용한 뒤 결과를 확인합니다.

from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 모델입니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터를 나누는 함수입니다.
from sklearn.metrics import accuracy_score  # 정확도를 계산하는 함수입니다.

# 원본 데이터
iris = load_iris()  # 예제 데이터셋을 불러옵니다.
X,y = iris.data, iris.target  # 모델에 넣을 입력 데이터입니다.


# PCA 적용
pca = PCA(n_components=2)  # PCA 차원축소입니다. n_components는 남길 요약 특성 수입니다.
X_pca = pca.fit_transform(X_scaled)  # PCA를 적용해 데이터를 2차원으로 바꿉니다.

# LDA 적용
lda = LDA(n_components=2)  # LDA입니다. 라벨 정보를 활용해 차원을 줄입니다.
X_lda = lda.fit_transform(X, y)  # LDA를 적용해 데이터를 2차원으로 바꿉니다.

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.
lr = LogisticRegression(max_iter=200)  # 로지스틱 회귀입니다. max_iter는 최대 반복입니다.
lr.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.
print("원본 데이터 정확도:", accuracy_score(y_test, lr.predict(X_test)))  # 학습한 모델로 새 값을 예측합니다.

# PCA 데이터
X_pca_train, X_pca_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.
lr.fit(X_pca_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.
print("PCA 데이터 정확도:", accuracy_score(y_test, lr.predict(X_pca_test)))  # 학습한 모델로 새 값을 예측합니다.

# LDA 데이터
X_lda_train, X_lda_test, y_train, y_test = train_test_split(X_lda, y, test_size=0.2, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.
lr.fit(X_lda_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.
print("LDA 데이터 정확도:", accuracy_score(y_test, lr.predict(X_lda_test)))  # 학습한 모델로 새 값을 예측합니다.

```
</details>



In [ ]:
# 여기에 코드를 작성하세요 
from sklearn.linear_model import LogisticRegression  # 로지스틱 회귀 분류 모델입니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.metrics import accuracy_score  # 정확도 지표 함수입니다.

# 원본 데이터
iris = load_iris()
X,y = iris.data, iris.target

### 문제 5. 피처 선택 vs 차원 축소
피처 선택과 차원 축소의 차이를 간단히 설명하세요.  

<details>
<summary>정답 보기</summary>

- **피처 선택 (Feature Selection)**: 원래의 특성 중 중요한 일부만 선택합니다. 해석이 쉽습니다.  
- **차원 축소 (Dimensionality Reduction)**: 여러 특성을 섞어 새로운 요약 특성을 만듭니다. 데이터 구조를 간단히 볼 수 있습니다.  
</details>


### 체크포인트
- PCA는 데이터 전체 모양을 보기 좋게 정리한다.  
- LDA는 클래스 구분이 잘 보이도록 정리한다.  
- 차원 축소 후에도 모델 성능을 반드시 재평가해야 한다.  
- 피처 선택과 차원 축소는 서로 다른 접근 방식이다.  


### 미니 프로젝트 : 상수관로 누수 여부 예측

**목표 :**
차원축소를 활용하여 데이터 전처리 후 성능 비교 해보기

**주제 :** 상수관로에 설치된 누수 감지 센서 데이터를 활용하여 **누수 여부를** 5가지 클래스로 분류

**문제 설명 :**  

제공된 데이터를 기반으로 **누수 여부(`leaktype`)를 예측**하는 AI 모델을 개발  
**다중 클래스 분류(multi-class classification)** 형태이며, 출력값은 **문자열(string)** 형태의 5개 클래스 중 하나  

**데이터 설명 :**

| 항목 | 설명 | 항목 | 설명 |
|------|------|------|------|
| `site` | 사이트 번호 | `sid` | 센서 번호 |
| `ldate` | 누수 감지 일자 | `lrate` | 누수 확률 |
| `llevel` | 누수 레벨 | `leaktype` | 누수 감지 클래스 (**예측 대상**) |
| `0Hz ~ 5120Hz` | 주파수별 감지된 누수 진동 크기 | `MAX~` | 감지 횟수 중 최대 주파수 및 최대 누수 크기 |

**예측 대상 (Target: `leaktype`)**

| 클래스명 | 의미 |
|-----------|------|
| `out` | 옥외 누수 |
| `in` | 옥내 누수 |
| `noise` | 기계·전기음 |
| `other` | 환경음 |
| `normal` | 정상음 |

**평가 지표**
- **Accuracy (정확도)** : 모델이 전체 데이터 중 정답을 맞춘 비율로 평가합니다.  

**활용 가능성**  
- 본 데이터는 **상수관로의 누수 및 파손 감지**, **상태 진단**, **스마트 워터 관리 시스템 개발**에 활용될 수 있습니다.

**데이터 출처**
- AI Hub — [상수관로 누수 감지 데이터 기반 AI 학습용 데이터셋](https://aihub.or.kr/ai../data/27709)

In [ ]:
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.
import numpy as np  # 배열과 수치 연산 라이브러리입니다.
import seaborn as sns  # 통계 그래프 시각화 도구입니다.
import matplotlib.pyplot as plt  # 그래프를 그리는 시각화 도구입니다.

from sklearn import preprocessing  # 사이킷런 도구를 불러옵니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix  # 정확도 지표 함수, 혼동행렬 함수입니다.
from sklearn.decomposition import PCA  # 차원축소 도구입니다.

import warnings
warnings.filterwarnings("ignore", category=UserWarning)  # XGBoost 실행 중 표시되는 UserWarning을 숨깁니다.
from xgboost import XGBClassifier  # XGBoost 분류 모델입니다.

In [ ]:
# 0. 데이터 불러오기
df = pd.read_csv('data/1.누수감지데이터-통합(leaks-all).csv')  # CSV 파일을 데이터프레임으로 읽습니다.
df.shape

In [ ]:
# 주파수 특성 컬럼만 추출
freq_cols = [c for c in df.columns if c.endswith("HZ") or c.startswith("MAX")]
df_freq = df[freq_cols]

In [ ]:
# 타깃 인코딩
le = preprocessing.LabelEncoder()  # 범주 라벨을 숫자로 바꿉니다.
df['leaktype_enc'] = le.fit_transform(df['leaktype'])  # 기준 학습과 변환을 한 번에 수행합니다.

In [ ]:
# 1. EDA
print("데이터 크기:", df.shape)  # 문자열을 정수로 바꿉니다.
print("타깃 분포:\n", df['leaktype'].value_counts())  # 값별 개수를 확인합니다.


In [ ]:
# 타깃 분포 시각화
sns.countplot(x='leaktype', data=df)  # 시본 그래프를 그립니다.
plt.title("Leaktype Distribution")  # 그래프 제목을 설정합니다.
plt.show()  # 그래프를 화면에 출력합니다.

In [ ]:
# 주파수 특성 기초 통계
print(df_freq.describe().T.head(10))  # 기초 통계량을 확인합니다.

# 결측치 확인
print("결측치 수:\n", df.isnull().sum().sum())  # 문자열을 정수로 바꿉니다.

In [ ]:
# 2. 차원축소 전 모델 (원본 feature 사용)
X_train, X_test, y_train, y_test = train_test_split(  # 데이터를 훈련/평가용으로 나눕니다.
    df_freq, df['leaktype_enc'], test_size=0.2, random_state=42, stratify=df['leaktype_enc']
)

In [ ]:
model_orig = XGBClassifier()  # XGBoost 모델입니다. 트리 수와 학습률을 정합니다.
model_orig.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.
pred_orig = model_orig.predict(X_test)  # 학습한 모델로 새 값을 예측합니다.

acc_orig = accuracy_score(y_test, pred_orig)  # 정확도 비율을 계산합니다.
f1_orig = f1_score(y_test, pred_orig, average="macro")  # F1 점수를 계산합니다.
print("차원축소 전 - Accuracy:", acc_orig, "F1:", f1_orig)  # 문자열을 정수로 바꿉니다.

In [ ]:
# 3. 차원축소 후 모델 (PCA 적용)
n_components = 100
pca = PCA(n_components=n_components)  # PCA 차원축소입니다. n_components는 줄일 차원 수입니다.
X_pca = pca.fit_transform(df_freq)  # 기준 학습과 변환을 한 번에 수행합니다.

Xp_train, Xp_test, yp_train, yp_test = train_test_split(  # 데이터를 훈련/평가용으로 나눕니다.
    X_pca, df['leaktype_enc'], test_size=0.2, random_state=42, stratify=df['leaktype_enc']
)

In [ ]:
model_pca = XGBClassifier()  # XGBoost 모델입니다. 트리 수와 학습률을 정합니다.
model_pca.fit(Xp_train, yp_train)  # 데이터로 모델이나 변환 기준을 학습합니다.
pred_pca = model_pca.predict(Xp_test)  # 학습한 모델로 새 값을 예측합니다.

acc_pca = accuracy_score(yp_test, pred_pca)  # 정확도 비율을 계산합니다.
f1_pca = f1_score(yp_test, pred_pca, average="macro")  # F1 점수를 계산합니다.
print("차원축소 후(PCA) - Accuracy:", acc_pca, "F1:", f1_pca)  # 문자열을 정수로 바꿉니다.

In [ ]:
# Confusion Matrix 비교 (차원축소 전 vs 후)
from sklearn.metrics import confusion_matrix  # 혼동행렬 함수입니다.


fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 차원축소 전
cm_orig = confusion_matrix(y_test, pred_orig)  # 실제값과 예측값의 혼동행렬입니다.
sns.heatmap(pd.DataFrame(cm_orig, index=le.classes_, columns=le.classes_),  # 시본 그래프를 그립니다.
            annot=True, fmt="d", cmap="Blues", ax=axes[0])

axes[0].set_title("Before PCA", fontsize=14)
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

# 차원축소 후 (PCA)
cm_pca = confusion_matrix(yp_test, pred_pca)  # 실제값과 예측값의 혼동행렬입니다.
sns.heatmap(pd.DataFrame(cm_pca, index=le.classes_, columns=le.classes_),  # 시본 그래프를 그립니다.
            annot=True, fmt="d", cmap="viridis", ax=axes[1])
axes[1].set_title("After PCA", fontsize=14)
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

plt.tight_layout()
plt.show()  # 그래프를 화면에 출력합니다.


In [ ]:
dff = pd.DataFrame(cm_orig, index=le.classes_, columns=le.classes_)  # 데이터프레임을 직접 만듭니다.

### 정리  

- 특성이 줄어들면 모델이 처리할 정보가 단순해질 수 있습니다.  
- 하지만 너무 많이 줄이면 필요한 신호까지 사라질 수 있습니다.  

**적정 차원 수는 어떻게 정할까?**  
정답은 하나로 고정되어 있지 않습니다.  
가방을 너무 작게 고르면 필요한 물건을 못 넣고, 너무 크게 고르면 짐이 줄지 않는 것과 비슷합니다.

**좋은 접근 방법**
1. PCA로 여러 개의 차원 수를 시도합니다.  
2. 각 차원 수로 모델을 학습합니다.  
3. 검증셋에서 **정확도(Accuracy)**, **F1-score** 등을 비교합니다.  
4. 성능이 안정적이면서 특성 수가 줄어드는 지점을 선택합니다.

요약하면, PCA는 데이터를 더 작게 접어 보는 도구입니다.  
최종 차원 수는 그래프와 실제 검증 성능을 함께 보고 결정합니다.


## 캐글 제출 권장 실습

이번 장에서 다룬 분류 모델과 회귀 모델을 실제 Kaggle 제출 형식으로 마무리합니다. 아래 Baseline 코드는 제출 파일을 만드는 최소 흐름입니다. 그대로 한 번 제출해 본 뒤, 전처리와 모델을 직접 개선해서 점수를 올려 보세요.

### 분류문제 캐글: Titanic 생존 예측

- 대회 주소: https://www.kaggle.com/competitions/titanic/overview  
- 문제 유형: 생존 여부를 예측하는 **분류 문제**  
- 제출 파일: `submission.csv`  
- 권장 목표: 전처리와 모델을 개선해 스코어 0.82 이상을 목표로 제출해 보기  
- 주의사항: Kaggle 제출 횟수 제한이 있으므로 검증 성능을 먼저 확인한 뒤 제출하세요.


In [ ]:
# 필요한 라이브러리 임포트
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.ensemble import RandomForestClassifier  # 랜덤포레스트 분류 모델입니다.
from sklearn.metrics import accuracy_score, classification_report  # 정확도 지표 함수, 분류 지표 요약 함수입니다.

# 데이터 불러오기
train = pd.read_csv('titanic/train.csv')  # CSV 파일을 데이터프레임으로 읽습니다.
test = pd.read_csv('titanic/test.csv')  # CSV 파일을 데이터프레임으로 읽습니다.

# 데이터 전처리
def preprocess_data(df):
    # 결측치 처리
    df['Age'] = df['Age'].fillna(df['Age'].mean())  # 결측치를 지정한 값으로 채웁니다.
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])  # 결측치를 지정한 값으로 채웁니다.
    df['Fare'] = df['Fare'].fillna(df['Fare'].mean())  # 결측치를 지정한 값으로 채웁니다.
    
    # 범주형 변수 처리
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})  # 값을 지정한 규칙으로 바꿉니다.
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})  # 값을 지정한 규칙으로 바꿉니다.
    
    # 필요한 특성 선택
    features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
    return df[features]

# 학습 데이터 전처리
X = preprocess_data(train)
y = train['Survived']

# 학습 데이터와 검증 데이터 분리
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.

# RandomForest 모델 생성 및 학습
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)  # 랜덤포레스트입니다. n_estimators는 트리 수입니다.
rf_model.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.

# 검증 데이터로 예측
val_pred = rf_model.predict(X_val)  # 학습한 모델로 새 값을 예측합니다.

# 모델 성능 평가
print('검증 데이터 정확도:', accuracy_score(y_val, val_pred))  # 정확도 비율을 계산합니다.
print('\n분류 보고서:')  # 문자열을 정수로 바꿉니다.
print(classification_report(y_val, val_pred))  # 정밀도/재현율/F1을 요약합니다.

# 테스트 데이터 예측
test_processed = preprocess_data(test)
test_pred = rf_model.predict(test_processed)  # 학습한 모델로 새 값을 예측합니다.

# 제출 파일 생성
submission = pd.DataFrame({  # 데이터프레임을 직접 만듭니다.
    'PassengerId': test['PassengerId'],
    'Survived': test_pred
})
submission.to_csv('submission.csv', index=False)
print('\n제출 파일이 생성되었습니다.')  # 문자열을 정수로 바꿉니다.


### 회귀문제 캐글: 자전거 수요 예측

- 대회 주소: https://www.kaggle.com/competitions/bike-sharing-demand/overview  
- 문제 유형: 대여량(`count`)을 예측하는 **회귀 문제**  
- 제출 파일: `bike_submission.csv`  
- 권장 목표: 전처리와 모델을 개선해 RMSLE 0.8 이하를 목표로 제출해 보기  
- 참고: 아래 코드는 RandomForest Baseline입니다. TensorFlow 딥러닝 모델이나 XGBoost 등으로 바꿔 성능을 비교해 보세요.


In [ ]:
# 필요한 라이브러리 임포트
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.
import numpy as np  # 배열과 수치 연산 라이브러리입니다.
from sklearn.ensemble import RandomForestRegressor  # 사이킷런 도구를 불러옵니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.metrics import mean_squared_error, mean_absolute_error  # MSE 회귀 지표 함수입니다.

# 데이터 불러오기 
train = pd.read_csv('./자전거수요예측/train.csv')  # CSV 파일을 데이터프레임으로 읽습니다.
test = pd.read_csv('./자전거수요예측/test.csv')  # CSV 파일을 데이터프레임으로 읽습니다.

# datetime 컬럼을 datetime 타입으로 변환
train['datetime'] = pd.to_datetime(train['datetime'])  # 문자열 날짜를 datetime으로 바꿉니다.
test['datetime'] = pd.to_datetime(test['datetime'])  # 문자열 날짜를 datetime으로 바꿉니다.

# datetime에서 유용한 특성 추출
train['year'] = train['datetime'].dt.year
train['month'] = train['datetime'].dt.month
train['day'] = train['datetime'].dt.day
train['hour'] = train['datetime'].dt.hour
train['dayofweek'] = train['datetime'].dt.dayofweek

test['year'] = test['datetime'].dt.year
test['month'] = test['datetime'].dt.month
test['day'] = test['datetime'].dt.day
test['hour'] = test['datetime'].dt.hour
test['dayofweek'] = test['datetime'].dt.dayofweek

# 사용할 특성 선택
features = ['season', 'holiday', 'workingday', 'weather', 'temp', 
           'atemp', 'humidity', 'windspeed', 'year', 'month', 
           'day', 'hour', 'dayofweek']

X = train[features]
y = train['count']

# 학습/검증 데이터 분리
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.

# 랜덤 포레스트 모델 학습
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.

# 검증 데이터로 성능 평가
val_pred = rf_model.predict(X_val)  # 학습한 모델로 새 값을 예측합니다.
print('검증 데이터 RMSE:', np.sqrt(mean_squared_error(y_val, val_pred)))  # 평균제곱오차를 계산합니다.
print('검증 데이터 MAE:', mean_absolute_error(y_val, val_pred))  # 문자열을 정수로 바꿉니다.

# 테스트 데이터 예측
X_test = test[features]
test_pred = rf_model.predict(X_test)  # 학습한 모델로 새 값을 예측합니다.

# 제출 파일 생성
submission = pd.DataFrame({  # 데이터프레임을 직접 만듭니다.
    'datetime': test['datetime'],
    'count': test_pred
})
submission.to_csv('bike_submission.csv', index=False)

print('제출 파일이 생성되었습니다.')  # 문자열을 정수로 바꿉니다.
